# Statistics and `ANALYZE TABLE`

A Spark 3.5+ lab using Hive for catalog metadata and HDFS for table data. We begin with an empty, isolated database, create tables without statistics, inspect optimizer estimates, collect table and column statistics, and compare the plans again.

> Services expected on this machine: HDFS at `hdfs://localhost:9000` and Hive Metastore Thrift at `thrift://localhost:9083`.

## Learning goals

You will learn:

- what Spark statistics are—and what they are not;
- where Hive metadata ends and HDFS data begins;
- how to inspect a table before statistics exist;
- what `ANALYZE TABLE ... NOSCAN`, table analysis, partition analysis, and column analysis collect;
- how Catalyst's cost-based optimizer (CBO) and join planner consume estimates;
- why catalog row counts do not normally make `DataFrame.count()` free or approximate.

# 1. Architecture: Hive metadata, HDFS bytes

The Hive Metastore stores database/table definitions, locations, partitions, and collected statistics. HDFS stores Parquet data files. Spark reads the catalog through the metastore and reads table bytes through Hadoop.

```text
DataFrame / Spark SQL
        |
        +--> Catalyst analyzer + optimizer + CBO
        |           |
        |           +--> Hive Metastore (localhost:9083): schema, location, statistics
        |
        +--------------> HDFS (localhost:9000): warehouse and external Parquet files
```

Statistics are advisory metadata used to choose a plan. They are not indexes, constraints, cached answers, or a replacement for reading data when an exact query result is required.

# 2. Spark session with Hive support

Hive support must be enabled while creating the session. If this notebook is attached to an incompatible existing session, stop that session and rerun from the top; `enableHiveSupport()` cannot retrofit a different catalog implementation into an already-created `SparkSession`.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from time import perf_counter

HDFS = "hdfs://localhost:9000"
METASTORE = "thrift://localhost:9083"
WAREHOUSE = f"{HDFS}/user/hive/warehouse"
DB = "statistics_analysis_lab"
EXTERNAL_ROOT = f"{HDFS}/tmp/dataeng/statistics_analysis"

spark = (
    SparkSession.builder.appName("Statistics-Analyse")
    .master("local[*]")
    .config("spark.sql.catalogImplementation", "hive")
    .config("hive.metastore.uris", METASTORE)
    .config("spark.hadoop.fs.defaultFS", HDFS)
    .config("spark.sql.warehouse.dir", WAREHOUSE)
    .config("spark.sql.cbo.enabled", "true")
    .config("spark.sql.cbo.joinReorder.enabled", "true")
    .config("spark.sql.cbo.planStats.enabled", "true")
    .config("spark.sql.statistics.size.autoUpdate.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Catalog implementation:", spark.conf.get("spark.sql.catalogImplementation"))
print("Default FS:", spark.sparkContext._jsc.hadoopConfiguration().get("fs.defaultFS"))
print("Warehouse:", spark.conf.get("spark.sql.warehouse.dir"))
print("Metastore:", spark.conf.get("hive.metastore.uris"))


## Short note: What is CBO, and when does it work?

**CBO means Cost-Based Optimizer.** It works **before query execution**, not after the final physical plan has run. Spark uses catalog statistics while optimizing the logical plan and while comparing possible physical strategies. It estimates the cost of alternatives and selects the plan it expects to be cheaper.

CBO helps Spark decide:

- the order in which multiple inner joins should happen;
- how many rows may remain after a filter;
- the estimated number of rows and bytes passed between operators;
- whether a relation appears small enough for a broadcast join;
- which candidate plan is likely to require less processing.

```text
SQL / DataFrame
      -> analyzed logical plan
      -> optimized logical plan  <-- CBO uses table and column statistics here
      -> physical-plan selection <-- size estimates also influence join strategy
      -> execution               <-- AQE may adapt remaining work at runtime
```

CBO depends on useful, current statistics. Without `ANALYZE TABLE`, Spark often relies on file sizes, default estimates, and heuristic rules. With table and column statistics, it can make better-informed estimates. **AQE is different:** CBO plans before execution using catalog estimates, whereas Adaptive Query Execution can revise parts of the physical plan after execution has started and runtime statistics become available.

## Connection preflight

This cell performs small read-only checks before the destructive lab reset. If it fails, confirm NameNode port 9000, Metastore port 9083, Hadoop/Hive client JAR compatibility, and filesystem permissions.

In [ ]:
spark.sql("SHOW DATABASES").show(truncate=False)
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
print("HDFS URI seen by Hadoop:", fs.getUri().toString())


# 3. Start from a plain slate

The next cell drops only the dedicated `statistics_analysis_lab` database. `CASCADE` removes its table metadata; managed table data below its warehouse database directory is also removed according to catalog semantics. It does not remove the separate external-data directory. We delete that dedicated external directory with Hadoop's filesystem API after validating the exact URI.

> Do not rename `DB` or `EXTERNAL_ROOT` to a shared production location.

In [ ]:
assert DB == "statistics_analysis_lab"
assert EXTERNAL_ROOT == "hdfs://localhost:9000/tmp/dataeng/statistics_analysis"

spark.sql(f"DROP DATABASE IF EXISTS {DB} CASCADE")
external_root_path = spark._jvm.org.apache.hadoop.fs.Path(EXTERNAL_ROOT)
if fs.exists(external_root_path):
    fs.delete(external_root_path, True)

spark.sql(f"CREATE DATABASE {DB} LOCATION '{WAREHOUSE}/{DB}.db'")
spark.sql(f"USE {DB}")
print("Tables at the start of the lab:")
spark.sql("SHOW TABLES").show(truncate=False)


# 4. Create data and tables—with analysis disabled

We deliberately keep `spark.sql.statistics.size.autoUpdate.enabled=false`. This prevents Spark from automatically refreshing catalog size statistics after writes, making the before/after analysis visible.

Three tables provide a meaningful three-way join-reordering example:

- `fact_events`: external, partitioned Parquet in `/tmp/dataeng/statistics_analysis`;
- `dim_customers`: managed Parquet in the Hive warehouse;
- `dim_regions`: a small managed Parquet dimension.

External means dropping the table normally leaves the external files. Managed means Hive/Spark owns the warehouse data lifecycle. Both table definitions and their statistics live in Hive Metastore.

In [ ]:
regions_df = spark.createDataFrame(
    [(0, "South"), (1, "West"), (2, "North"), (3, "East")], ["region_id", "region_name"]
)

customers_df = spark.range(0, 20_000, numPartitions=8).select(
    F.col("id").alias("customer_id"),
    (F.col("id") % 4).cast("int").alias("region_id"),
    F.when(F.col("id") < 400, "VIP").otherwise("STANDARD").alias("segment"),
    F.concat(F.lit("customer_"), F.col("id")).alias("customer_name"),
)

events_df = (
    spark.range(0, 600_000, numPartitions=24)
    .select(
        F.col("id").alias("event_id"),
        (F.col("id") % 20_000).alias("customer_id"),
        F.date_add(
            F.lit("2025-01-01").cast("date"), (F.col("id") % 365).cast("int")
        ).alias("event_date"),
        F.element_at(
            F.array(F.lit("view"), F.lit("cart"), F.lit("purchase")),
            (F.col("id") % 3 + 1).cast("int"),
        ).alias("event_type"),
        F.round((F.col("id") % 1000) * 0.41, 2).alias("amount"),
    )
    .withColumn("event_year", F.year("event_date"))
    .withColumn("event_month", F.month("event_date"))
)

regions_df.write.mode("overwrite").format("parquet").saveAsTable("dim_regions")
customers_df.write.mode("overwrite").format("parquet").saveAsTable("dim_customers")

events_path = f"{EXTERNAL_ROOT}/fact_events"
(
    events_df.write.mode("overwrite")
    .partitionBy("event_year", "event_month")
    .parquet(events_path)
)
spark.sql(f"""
CREATE TABLE fact_events (
  event_id BIGINT, customer_id BIGINT, event_date DATE,
  event_type STRING, amount DOUBLE, event_year INT, event_month INT
) USING PARQUET
PARTITIONED BY (event_year, event_month)
LOCATION '{events_path}'
""")
spark.sql("MSCK REPAIR TABLE fact_events")
spark.sql("SHOW TABLES").show(truncate=False)


# 5. Inspect statistics before `ANALYZE`

## What to inspect

- `DESCRIBE EXTENDED table`: table type, location, provider, partitioning, and possibly a `Statistics` row;
- `DESCRIBE EXTENDED table column`: column-stat fields, usually `NULL` before column analysis;
- `EXPLAIN COST`: optimizer `sizeInBytes` and, when known, `rowCount`;
- the JVM optimized plan stats: useful for a compact programmatic comparison.

Important: an absent catalog `Statistics` row does not mean Spark knows nothing. A file scan can infer a size from HDFS file metadata. It generally does not know an exact row count or column distribution until those statistics are collected. Output differs slightly by Spark/Hive versions.

In [ ]:
def describe_statistics(table_name):
    print(f"\n--- DESCRIBE EXTENDED {table_name} ---")
    details = spark.sql(f"DESCRIBE EXTENDED {table_name}")
    details.where(
        F.col("col_name").isin("Type", "Provider", "Location", "Statistics")
    ).show(truncate=False)


def optimizer_stats(df, label):
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    row_count = (
        str(stats.rowCount().get()) if stats.rowCount().isDefined() else "UNKNOWN"
    )
    print(f"{label}: sizeInBytes={stats.sizeInBytes()}, rowCount={row_count}")


for table in ["fact_events", "dim_customers", "dim_regions"]:
    describe_statistics(table)
    optimizer_stats(spark.table(table), table)

spark.sql("DESCRIBE EXTENDED fact_events amount").show(truncate=False)


## Baseline query and cost plan

The CBO (`CostBasedJoinReorder`) can reorder inner joins using row counts and column statistics. The physical join planner also uses relation-size estimates when considering broadcast joins. Filter selectivity estimation improves when min/max, null counts, and distinct counts are known.

We disable AQE in this comparison so runtime adaptations do not obscure the catalog-statistics lesson. AQE is a separate runtime optimizer; it can revise parts of the physical plan using observed shuffle statistics even when catalog estimates are missing or stale.

In [ ]:
baseline_df = (
    spark.table("fact_events")
    .alias("f")
    .join(spark.table("dim_customers").alias("c"), "customer_id")
    .join(spark.table("dim_regions").alias("r"), "region_id")
    .where(
        (F.col("f.event_year") == 2025)
        & (F.col("f.event_month") == 6)
        & (F.col("c.segment") == "VIP")
    )
    .groupBy("region_name")
    .agg(F.sum("amount").alias("revenue"))
)

print("BEFORE ANALYZE")
baseline_df.explain(mode="cost")
baseline_df.explain(mode="formatted")


# 6. `ANALYZE TABLE ... NOSCAN`

### Statistics operation
`ANALYZE TABLE table COMPUTE STATISTICS NOSCAN`

### What it does
It updates table-size metadata without scanning every row. For file-based tables, size comes from file metadata/listing. It is relatively cheap, but it does not collect a reliable row count and does not collect column distributions.

### How the optimizer uses it
Estimated bytes can affect broadcast eligibility and cost comparisons. It is less useful than full statistics for estimating filter cardinality and multi-join costs.

### Deeper topic
Run `NOSCAN` when byte size is enough and a data scan is too expensive. On partitioned tables, metadata listing itself can still be costly when there are very many partitions/files.

In [ ]:
for table in ["fact_events", "dim_customers", "dim_regions"]:
    spark.sql(f"ANALYZE TABLE {table} COMPUTE STATISTICS NOSCAN")
    describe_statistics(table)
    optimizer_stats(spark.table(table), f"{table} after NOSCAN")


# 7. Full table statistics

### Statistics operation
`ANALYZE TABLE table COMPUTE STATISTICS`

### What it does
Spark scans the table to compute table-level statistics, notably total bytes and row count. This costs a read job but gives the optimizer a real cardinality baseline. It does not automatically create all column statistics.

### How the optimizer uses it
Row count lets CBO estimate how many rows flow through joins and aggregates. Together with sizes, it helps compare alternative join orders and strategies.

### Deeper topic
Statistics become stale after inserts, overwrites, or changing external files. Analysis is an operational maintenance task: collect it after significant changes, but avoid repeatedly scanning unchanged large tables.

In [ ]:
for table in ["fact_events", "dim_customers", "dim_regions"]:
    spark.sql(f"ANALYZE TABLE {table} COMPUTE STATISTICS")
    describe_statistics(table)
    optimizer_stats(spark.table(table), f"{table} after full table analysis")


# 8. Column statistics

### Statistics operation
`ANALYZE TABLE table COMPUTE STATISTICS FOR COLUMNS col1, col2, ...`

### What it does
For supported data types Spark collects values such as distinct count, minimum, maximum, null count, average length, maximum length, and sometimes a histogram when `spark.sql.statistics.histogram.enabled=true`. The command scans the selected columns.

### How the optimizer uses it
CBO estimates predicate selectivity and join cardinality. For example, distinct counts on join keys and min/max or histograms on filtered columns can distinguish a selective predicate from one that retains most rows.

### Deeper topic
Analyze columns that participate in joins, filters, and grouping—not every wide-table column by habit. Distinct-count sketches and histograms are estimates. Correlation between columns is usually not captured, so combined-predicate estimates can still be wrong.

In [ ]:
spark.sql(
    "ANALYZE TABLE fact_events COMPUTE STATISTICS FOR COLUMNS customer_id, event_type, amount"
)
spark.sql(
    "ANALYZE TABLE dim_customers COMPUTE STATISTICS FOR COLUMNS customer_id, region_id, segment"
)
spark.sql("ANALYZE TABLE dim_regions COMPUTE STATISTICS FOR COLUMNS region_id")

print("Column statistics after analysis:")
spark.sql("DESCRIBE EXTENDED fact_events amount").show(truncate=False)
spark.sql("DESCRIBE EXTENDED dim_customers segment").show(truncate=False)
spark.sql("DESCRIBE EXTENDED dim_customers customer_id").show(truncate=False)


# 9. Partition statistics

### Statistics operation
`ANALYZE TABLE table PARTITION (...) COMPUTE STATISTICS`

### What it does
It refreshes statistics for a specific Hive partition rather than the entire partitioned table. This is useful after incrementally loading one date/month partition.

### How the optimizer uses it
When partition pruning selects that partition, its statistics can improve scan/cardinality estimates. The partition must be registered in the metastore; `MSCK REPAIR TABLE` registered the externally written directories earlier.

### Deeper topic
Prefer targeted analysis after incremental loads. Table-level statistics may still need refresh when global totals materially change. Partition discovery and statistics collection solve different problems.

In [ ]:
spark.sql("""
ANALYZE TABLE fact_events
PARTITION (event_year=2025, event_month=6)
COMPUTE STATISTICS
""")
spark.sql(
    "DESCRIBE EXTENDED fact_events PARTITION (event_year=2025, event_month=6)"
).where(F.col("col_name").isin("Location", "Statistics", "Partition Statistics")).show(
    truncate=False
)


# 10. Compare the optimizer after analysis

Rebuild the DataFrame so analysis starts with current catalog metadata. Compare `rowCount`, `sizeInBytes`, filter estimates, join order, build side, and join algorithm with the saved baseline output. Small demo tables may produce the same physical strategy before and after; that is a valid result. The important change is that costs are based on real metadata rather than fallbacks.

Use `explain(mode='cost')` for estimates and `formatted` for physical structure. A plan change is not proof of improvement—confirm runtime metrics in the Spark UI.

In [ ]:
after_analyze_df = (
    spark.table("fact_events")
    .alias("f")
    .join(spark.table("dim_customers").alias("c"), "customer_id")
    .join(spark.table("dim_regions").alias("r"), "region_id")
    .where(
        (F.col("f.event_year") == 2025)
        & (F.col("f.event_month") == 6)
        & (F.col("c.segment") == "VIP")
    )
    .groupBy("region_name")
    .agg(F.sum("amount").alias("revenue"))
)

print("AFTER TABLE + COLUMN ANALYZE")
after_analyze_df.explain(mode="cost")
after_analyze_df.explain(mode="formatted")
after_analyze_df.show(truncate=False)


# 11. SQL and DataFrame are optimized by the same engine

DataFrame transformations construct logical plans; SQL text is parsed into logical plans. After analysis, both pass through Catalyst and the same physical planner. Equivalent expressions should normally converge to equivalent optimized plans.

In [ ]:
sql_df = spark.sql("""
SELECT r.region_name, SUM(f.amount) AS revenue
FROM fact_events f
JOIN dim_customers c ON f.customer_id = c.customer_id
JOIN dim_regions r ON c.region_id = r.region_id
WHERE f.event_year = 2025 AND f.event_month = 6 AND c.segment = 'VIP'
GROUP BY r.region_name
""")
sql_df.explain(mode="cost")
sql_df.show(truncate=False)


# 12. What exactly happens with `count()`?

Catalog `rowCount` is an optimizer estimate. In ordinary Spark table queries, `df.count()` still builds and executes an aggregate to produce an exact current result; Spark cannot safely return a possibly stale catalog number. `ANALYZE TABLE` therefore does **not** turn arbitrary counts into constant-time metadata lookups.

For columnar formats, Spark can often avoid decoding unneeded columns for `count(*)`, and some source/version/configuration combinations support aggregate pushdown using file metadata. Verify the actual physical plan: do not assume it. `count(column)` must also respect nulls; filtered counts must evaluate the filter.

The cell prints the catalog estimate without triggering a Spark action, then explains and executes an exact count. The private JVM access is appropriate for learning/diagnostics but is not a stable application API.

In [ ]:
fact_df = spark.table("fact_events")
optimizer_stats(fact_df, "Catalog/optimizer estimate (no action)")

exact_count_plan = fact_df.agg(F.count(F.lit(1)).alias("exact_rows"))
exact_count_plan.explain(mode="formatted")
started = perf_counter()
exact_rows = exact_count_plan.first()["exact_rows"]
print(f"Exact count={exact_rows:,}; elapsed={perf_counter() - started:.3f}s")

filtered_count = fact_df.where(F.col("amount") >= 300).count()
print(f"Exact filtered count={filtered_count:,}")


# 13. Related commands and settings

| Feature | Purpose | Important limitation |
|---|---|---|
| `DESCRIBE EXTENDED/FORMATTED` | Inspect table, partition, and column metadata | Presentation varies by version/provider |
| `EXPLAIN COST` | Show estimated sizes/cardinalities | Estimates can be absent or stale |
| `ANALYZE ... NOSCAN` | Refresh byte size cheaply | No row or column distribution |
| `ANALYZE ... COMPUTE STATISTICS` | Collect table bytes and row count | Requires a scan |
| `ANALYZE ... FOR COLUMNS` | Collect NDV/min/max/null/length information | Scan cost; correlation usually unknown |
| `ANALYZE ... PARTITION` | Refresh one loaded partition | Does not automatically refresh every global statistic |
| `MSCK REPAIR TABLE` | Register filesystem partitions | Does not analyze their data |
| `REFRESH TABLE` | Invalidate cached table/file metadata | Does not collect CBO statistics |
| `CACHE TABLE` / `df.persist()` | Reuse computed data | Runtime cache, not catalog statistics |
| `spark.sql.statistics.size.autoUpdate.enabled` | Auto-update size after data changes | Extra overhead; not full column statistics |
| AQE | Adapt using runtime shuffle evidence | Separate from catalog CBO statistics |

# 14. Statistics maintenance workflow

A production-friendly pattern is:

1. Load or overwrite data and register new partitions.
2. Refresh statistics only for materially changed tables/partitions.
3. Collect column statistics for frequently joined and filtered columns.
4. Inspect `EXPLAIN COST` and physical plans for important queries.
5. Validate estimates against runtime rows, skew, shuffle, spill, and duration in the Spark UI.
6. Reanalyze when data volume or distribution changes enough to affect decisions.

More statistics are not always better: collection consumes cluster I/O and old statistics can be actively misleading. Focus maintenance on decision-relevant tables and columns.

## Optional cleanup

The lab leaves metadata and data available for inspection. The following cell is intentionally commented. Dropping the external table does not normally delete its HDFS files, so catalog and external storage cleanup are separate operations.

In [ ]:
# Uncomment only when finished with the lab. Exact-name assertions limit the scope.
# assert DB == "statistics_analysis_lab"
# assert EXTERNAL_ROOT == "hdfs://localhost:9000/tmp/dataeng/statistics_analysis"
# spark.sql("USE default")
# spark.sql(f"DROP DATABASE IF EXISTS {DB} CASCADE")
# p = spark._jvm.org.apache.hadoop.fs.Path(EXTERNAL_ROOT)
# if fs.exists(p):
#     fs.delete(p, True)
# spark.stop()
